- This script uses Sentence-Transformers for the embeddings and scikit-learn for the similarity calculation (simulating a local vector store for simplicity).

In [0]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Initialize the Encoder (using a lightweight, fast model)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define and Embed Taxonomy
# We define labels and 'anchor phrases' to give the vector space more context
taxonomy = {
    "sales": "inquiry about pricing, discounts, purchasing products, or enterprise plans",
    "customer_support": "technical issues, bug reports, or how-to questions about the platform",
    "account_lock": "forgot password, locked out of account, login issues, or 2FA problems"
}

labels = list(taxonomy.keys())
taxonomy_embeddings = model.encode(list(taxonomy.values()))

def semantic_router(user_input):
    # 3. User Input Analysis (Embedding)
    input_embedding = model.encode([user_input])

    # 4. Vector Search (Cosine Similarity)
    # This calculates the distance between input and all taxonomy items
    similarities = cosine_similarity(input_embedding, taxonomy_embeddings)[0]

    # 5. Route Mapping
    highest_score_idx = np.argmax(similarities)
    best_label = labels[highest_score_idx]
    confidence = similarities[highest_score_idx]

    return best_label, confidence

# --- Testing the System ---
test_prompts = [
    "How much does the pro version cost?",
    "I've been locked out of my dashboard and need a password reset.",
    "The button on the settings page isn't clicking for me."
]

for prompt in test_prompts:
    route, score = semantic_router(prompt)
    print(f"Prompt: '{prompt}'")
    print(f"Routed to: [{route.upper()}] (Confidence: {score:.4f})\n")
